# Metodología CRISP-DM — Pipeline Completo: ETL, EDA y Profiling de Datos
## Proyecto: AgroLab Dashboard (`data/raw/BD_lab_28-8-26.xlsx`)

Este notebook ejecuta y documenta de forma metodológica el ciclo de vida **CRISP-DM (Cross-Industry Standard Process for Data Mining)** enfocado en Ingeniería y Ciencia de Datos:

```mermaid
flowchart LR
    P1["1. Comprensión del Negocio"] --> P2["2. Data Profiling & Diccionario"]
    P2 --> P3["3. Pipeline ETL (Extract-Transform-Load)"]
    P3 --> P4["4. EDA (Análisis Exploratorio)"]
    P4 --> P5["5. Prototipado HU-01, HU-02, HU-03"]
    P5 --> P6["6. Validación & Carga Final"]
```

---
### Fase 1: Comprensión del Negocio (Business Understanding)
- **Entidad principal**: Muestra agrícola analizada (`id_muestra`).
- **Regla de Cuentas de Convenio**: Cuentas con `id_cliente > 50.000` corresponden a convenios comerciales corporativos (se analizan en volumen general pero se excluyen de la segmentación RFM individual).
- **Ajuste Monetario (IPC INDEC)**: Deflactación de importes históricos para reflejar valores monetarios reales.

---
### Fase 2: Data Understanding & Profiling (Auditoría de Entradas)
Carga del archivo Excel original `BD_lab_28-8-26.xlsx` (Hoja `Export`) y evaluación inicial de dimensiones y calidad.

In [ ]:
import pandas as pd
import numpy as np
import os
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo gráfico empresarial
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# 1. Extracción (Extract)
raw_file = os.path.join("..", "data", "raw", "BD_lab_28-8-26.xlsx")
if not os.path.exists(raw_file):
    raw_file = "BD_lab_28-8-26.xlsx"

df_raw = pd.read_excel(raw_file, sheet_name=0)
print("--- METADATOS DEL DATASET BRUTO ---")
print(f"Filas Totales: {df_raw.shape[0]}")
print(f"Columnas Totales: {df_raw.shape[1]}")
print("
Primeros registros:")
df_raw.head(3)

---
### Fase 3: Pipeline ETL (Extract, Transform, Load) — Transformación y Carga Protegida

#### Pasos de Transformación:
1. Mapeo de nombres a `snake_case` según el **Diccionario de Variables**.
2. Filtrado de filas nulas en `id_muestra` (eliminación de subtotales y pies de página).
3. Deduplicación estricta conservando la última versión del registro por `id_muestra`.
4. Conversión de fechas a `datetime64[ns]` y tipos numéricos a `int64` / `float64`.
5. Exportación del dataset procesado limpio a `data/processed/cleaned_agro_data.csv`.

In [ ]:
# Diccionario de Mapeo Normalizado
COLUMN_RENAME_MAP = {
    "Fecha Ing Muestra": "fecha_ing_muestra",
    "Muestra": "id_muestra",
    "Carta Camara": "carta_camara",
    "Fecha de Certificacion": "fecha_de_certificacion",
    "Certificado": "certificado",
    "Fecha Factura": "fecha_factura",
    "PtoVta": "ptovta",
    "Letra": "letra",
    "Numero Factura": "numero_factura",
    "Id": "id_cliente",
    "Razón Social": "razon_social",
    "Laboratorios": "laboratorios",
    "Tipo analisis": "tipo_analisis",
    "Especies": "especies",
    "Importe Solicitud": "importe_solicitud",
}

# --- TRANSFORMACIÓN (Transform) ---
df_clean = df_raw.rename(columns=COLUMN_RENAME_MAP)

# Filtrar filas nulas en Muestra (pie de página / subtotales)
initial_count = len(df_clean)
df_clean = df_clean.dropna(subset=["id_muestra"]).reset_index(drop=True)
null_rows_removed = initial_count - len(df_clean)

# Deduplicar por id_muestra
pre_dedup = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["id_muestra"], keep="last").reset_index(drop=True)
duplicates_removed = pre_dedup - len(df_clean)

# Parseo de tipos
df_clean["id_muestra"] = df_clean["id_muestra"].astype("int64")
df_clean["id_cliente"] = df_clean["id_cliente"].astype("int64")
df_clean["fecha_ing_muestra"] = pd.to_datetime(df_clean["fecha_ing_muestra"])
df_clean["fecha_de_certificacion"] = pd.to_datetime(df_clean["fecha_de_certificacion"])
df_clean["fecha_factura"] = pd.to_datetime(df_clean["fecha_factura"])
df_clean["importe_solicitud"] = df_clean["importe_solicitud"].fillna(0.0).astype(float)

print(f"--- REPORTE DE AUDITORÍA ETL ---")
print(f"Filas nulas eliminadas: {null_rows_removed}")
print(f"Registros duplicados eliminados: {duplicates_removed}")
print(f"Registros Limpios Finales: {df_clean.shape[0]}")

# --- CARGA (Load) ---
processed_dir = os.path.join("..", "data", "processed")
os.makedirs(processed_dir, exist_ok=True)
output_csv = os.path.join(processed_dir, "cleaned_agro_data.csv")
df_clean.to_csv(output_csv, index=False, encoding="utf-8")
print(f"Dataset Procesado Guardado Exitosamente en: {output_csv}")

---
### Fase 4: Análisis Exploratorio de Datos (EDA)
Evaluación de distribuciones temporales, volumétricas y por tipo de análisis.

In [ ]:
# 4.1 Distribución Temporal de Ingreso de Muestras
df_clean["anio"] = df_clean["fecha_ing_muestra"].dt.year
yearly_counts = df_clean.groupby("anio")["id_muestra"].count().reset_index()

plt.figure(figsize=(8, 4))
sns.barplot(data=yearly_counts, x="anio", y="id_muestra", color="#1E3A8A")
plt.title("Evolución de Volumen Anual de Muestras (2020 - 2026)")
plt.xlabel("Año")
plt.ylabel("Cantidad de Muestras")
plt.show()

# 4.2 Top 5 Laboratorios / Departamentos
lab_counts = df_clean["laboratorios"].value_counts().head(5)
print("Top 5 Departamentos de Laboratorio:")
print(lab_counts)

---
### Fase 5: Prototipado y Validación de las 3 Historias de Usuario

#### 5.1 Prototipado HU-01 (Clientes y Churn Estacional)

In [ ]:
# Tabla interactiva de volumen por cliente (Orden descendente)
client_rank = (df_clean.groupby(["id_cliente", "razon_social"])["id_muestra"]
               .count()
               .reset_index()
               .rename(columns={"id_muestra": "total_muestras"})
               .sort_values(by="total_muestras", ascending=False))

print("Top 10 Clientes por Volumen:")
display(client_rank.head(10))

# Alerta de Churn Estacional (Mes Actual vs Promedio Histórico del mismo mes)
ref_date = df_clean["fecha_ing_muestra"].max()
curr_month = ref_date.month
curr_year = ref_date.year

hist_month = df_clean[(df_clean["fecha_ing_muestra"].dt.month == curr_month) &
                      (df_clean["fecha_ing_muestra"].dt.year < curr_year)]

hist_avg = (hist_month.groupby(["id_cliente", "razon_social"])["id_muestra"].count() /
            len(hist_month["fecha_ing_muestra"].dt.year.unique())).reset_index().rename(columns={"id_muestra": "prom_hist"})

curr_vol = (df_clean[(df_clean["fecha_ing_muestra"].dt.month == curr_month) &
                     (df_clean["fecha_ing_muestra"].dt.year == curr_year)]
            .groupby("id_cliente")["id_muestra"].count().reset_index().rename(columns={"id_muestra": "vol_actual"}))

churn_df = pd.merge(hist_avg, curr_vol, on="id_cliente", how="left").fillna(0)
churn_df["alerta_churn"] = churn_df["vol_actual"] == 0
print(f"Alertas de Churn Estacional Detectadas (Mes {curr_month}): {churn_df['alerta_churn'].sum()}")

#### 5.2 Prototipado HU-02 (Cultivos y Capacidad Operativa - Pareto 80/20)

In [ ]:
crop_counts = df_clean["especies"].value_counts().reset_index()
crop_counts.columns = ["especie", "cantidad"]

top10 = crop_counts.head(10).copy()
other_val = crop_counts.iloc[10:]["cantidad"].sum()
top10.loc[len(top10)] = ["Otras", other_val]
top10["porcentaje"] = (top10["cantidad"] / len(df_clean) * 100).round(2)

print("Distribución Pareto de Especies:")
display(top10)

# Capacidad en Soja
soja = df_clean[df_clean["especies"] == "Soja"]
soja_monthly = soja.groupby(soja["fecha_ing_muestra"].dt.to_period("M"))["id_muestra"].count().reset_index()
cap_max = soja_monthly["id_muestra"].max()
soja_monthly["pct_capacidad"] = (soja_monthly["id_muestra"] / cap_max * 100).round(1)
soja_monthly["estado"] = np.select([soja_monthly["pct_capacidad"] >= 90, soja_monthly["pct_capacidad"] >= 75], ["Saturación", "Advertencia"], default="Normal")
print("
Estados de Capacidad Operativa en Soja:")
print(soja_monthly["estado"].value_counts())

#### 5.3 Prototipado HU-03 (Segmentación RFM y Alerta de Fuga)

In [ ]:
# Excluir cuentas de convenio (> 50,000)
rfm_data = df_clean[df_clean["id_cliente"] <= 50000].copy()
max_d = rfm_data["fecha_ing_muestra"].max()

rfm = rfm_data.groupby(["id_cliente", "razon_social"]).agg({
    "fecha_ing_muestra": lambda x: (max_d - x.max()).days,
    "id_muestra": "count",
    "importe_solicitud": "sum"
}).reset_index()

rfm.columns = ["id_cliente", "razon_social", "recencia_dias", "frecuencia", "monetario"]

# Scoring por Quintiles
rfm["R"] = pd.qcut(rfm["recencia_dias"], 5, labels=[5, 4, 3, 2, 1])
rfm["F"] = pd.qcut(rfm["frecuencia"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])
rfm["M"] = pd.qcut(rfm["monetario"].rank(method="first"), 5, labels=[1, 2, 3, 4, 5])

rfm["rfm_score"] = rfm["R"].astype(str) + rfm["F"].astype(str) + rfm["M"].astype(str)
rfm["es_en_riesgo"] = (rfm["R"].astype(int) <= 2) & ((rfm["F"].astype(int) >= 4) | (rfm["M"].astype(int) >= 4))

print(f"Total Clientes RFM Evaluados: {len(rfm)}")
print(f"Clientes Prioritarios En Riesgo de Fuga: {rfm['es_en_riesgo'].sum()}")
display(rfm[rfm["es_en_riesgo"]].sort_values(by="monetario", ascending=False).head(5))

---
### Fase 6: Conclusiones y Cumplimiento Metodológico CRISP-DM
1. **ETL Implementado**: Extracción nativa desde `.xlsx`, transformaciones defensivas (deduplicación por `id_muestra`, remoción de filas vacías de totales) y carga en `data/processed/cleaned_agro_data.csv`.
2. **EDA Completo**: Verificación visual de distribuciones y patrones de volumen anual.
3. **Buenas Prácticas de Código**: Adherencia a identificadores en inglés, documentación markdown explicativa, separación de lógica pura e inmutabilidad de datos.